# Phase 0 — Real ML pipeline proof
F5-TTS → LivePortrait → MuseTalk in three isolated micromamba envs, producing one talking-head MP4.

Heavy stuff installs under `/tmp/aiavatar` (big disk). Only `output.mp4` lands in `/kaggle/working`.

**Settings → Accelerator: GPU T4 x2, Internet: ON.** Run cells top to bottom.

In [ ]:
# 1. Clone (or update) the repo to get the proof scripts.
import os, subprocess
os.chdir('/kaggle/working')
if os.path.exists('Ai-Avatar'):
    subprocess.run(['git','-C','Ai-Avatar','pull','--ff-only'], check=False)
else:
    subprocess.run(['git','clone','https://github.com/Yashvant203/Ai-Avatar.git'], check=True)
print('repo ready')

In [ ]:
# 2. Build envF5 (F5-TTS). ~8-12 min.
!bash /kaggle/working/Ai-Avatar/ml_models/proof/setup_env_f5.sh /tmp/aiavatar

In [ ]:
# 3. Build envLP (LivePortrait). ~8-12 min.
!bash /kaggle/working/Ai-Avatar/ml_models/proof/setup_env_lp.sh /tmp/aiavatar

In [ ]:
# 4. Build envMT (MuseTalk + mmcv). ~15-25 min. The riskiest install.
!bash /kaggle/working/Ai-Avatar/ml_models/proof/setup_env_mt.sh /tmp/aiavatar

In [ ]:
# 5. F5-TTS: clone a voice from F5's bundled reference, synthesize a test line.
import subprocess
M = ['bash','/kaggle/working/Ai-Avatar/ml_models/proof/mrun.sh','envF5']
REF = subprocess.check_output(M + ['python','-c',
    "from importlib.resources import files; print(files('f5_tts').joinpath('infer/examples/basic/basic_ref_en.wav'))"]).decode().strip()
subprocess.run(M + ['python','/kaggle/working/Ai-Avatar/ml_models/proof/run_f5tts.py','--ref',REF,'--ref-text','',
    '--gen-text','Hello, this is a test of my cloned voice on a free GPU.',
    '--out','/tmp/aiavatar/speech.wav'], check=True)
print('speech.wav written')

In [ ]:
# 6. Loop the driving clip to the speech length, then animate with LivePortrait.
import subprocess
dur = float(subprocess.check_output(['ffprobe','-v','error','-show_entries',
    'format=duration','-of','default=noprint_wrappers=1:nokey=1',
    '/tmp/aiavatar/speech.wav']).decode().strip())
print('speech duration', round(dur,2), 's')
drv_src = '/tmp/aiavatar/LivePortrait/assets/examples/driving/d0.mp4'
subprocess.run(['ffmpeg','-y','-stream_loop','-1','-i',drv_src,'-t',f'{dur:.2f}',
    '-r','25','/tmp/aiavatar/driving_loop.mp4'], check=True)
subprocess.run(['bash','/kaggle/working/Ai-Avatar/ml_models/proof/run_liveportrait.sh',
    '/tmp/aiavatar/LivePortrait/assets/examples/source/s9.jpg',
    '/tmp/aiavatar/driving_loop.mp4','/tmp/aiavatar/lp_out','/tmp/aiavatar'], check=True)
import glob; animated = [f for f in sorted(glob.glob('/tmp/aiavatar/lp_out/*.mp4')) if 'concat' not in f][-1]; print('animated:', animated)

In [ ]:
# 7. Normalize to MuseTalk inputs: 25fps video + 16kHz mono audio.
import glob, subprocess
animated = [f for f in sorted(glob.glob('/tmp/aiavatar/lp_out/*.mp4')) if 'concat' not in f][-1]
subprocess.run(['ffmpeg','-y','-i',animated,'-r','25','/tmp/aiavatar/animated_25.mp4'], check=True)
subprocess.run(['ffmpeg','-y','-i','/tmp/aiavatar/speech.wav','-ar','16000','-ac','1','/tmp/aiavatar/speech_16k.wav'], check=True)
print('normalized inputs ready')

In [ ]:
# 8. MuseTalk lip-sync (envMT), from the MuseTalk repo root.
import subprocess
subprocess.run(['bash','/kaggle/working/Ai-Avatar/ml_models/proof/mrun.sh','envMT','python',
    '/kaggle/working/Ai-Avatar/ml_models/proof/run_musetalk.py',
    '--video','/tmp/aiavatar/animated_25.mp4',
    '--audio','/tmp/aiavatar/speech_16k.wav',
    '--result-dir','/tmp/aiavatar/mt_out'], cwd='/tmp/aiavatar/MuseTalk', check=True)
import glob; print('musetalk out:', glob.glob('/tmp/aiavatar/mt_out/**/*.mp4', recursive=True))

In [ ]:
# 9. Mux cloned speech onto the lip-synced video → /kaggle/working/output.mp4.
import glob, subprocess
lip = glob.glob('/tmp/aiavatar/mt_out/**/*.mp4', recursive=True)[-1]
subprocess.run(['ffmpeg','-y','-i',lip,'-i','/tmp/aiavatar/speech.wav',
    '-c:v','copy','-c:a','aac','-shortest','/kaggle/working/output.mp4'], check=True)
from IPython.display import Video; Video('/kaggle/working/output.mp4', embed=True, width=512)

## Done
If `output.mp4` shows the face speaking with plausible motion + lip-sync, the stack is proven.

### Use your own face + voice
In cell 5 set `--ref` to a ~10s clean wav of you (and `--ref-text` to its transcript); in cell 6 set the source image to your face photo.

### Cache weights
Save `/tmp/aiavatar/LivePortrait/pretrained_weights`, `/tmp/aiavatar/MuseTalk/models`, and the F5-TTS HF cache as a Kaggle Dataset, then mount it next session and skip the download steps.